# Nemotron-3-Nano-30B v0.5 SFT — Submission Demo

Loads the pre-trained v0.5 LoRA adapter and writes it to `/kaggle/working` for
competition submission.

**Submission approach:** The adapter is provided as a Kaggle dataset input
(`gdataranger/nemotron-v05-sft-unsloth`). No base model loading or internet access required —
the competition evaluator loads the base model and adapter separately. This notebook
only needs to produce the adapter files in `/kaggle/working`.

Training was done off-Kaggle on a GB10 (DGX Spark) using Unsloth for joint MoE+attention
LoRA training. See the
[prize eligibility notebook](https://www.kaggle.com/code/gdataranger/nemotron-3-nano-30b-lora-reasoning-challenge)
for full methodology.

**How we actually submit (off-Kaggle):**
```bash
bash scripts/package_submission.sh output/adapter_v5_sft_unsloth output/submission_v5_unsloth
kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f output/submission_v5_unsloth/submission.zip \
  -m "v0.5 SFT Unsloth — warmstart v27 + 240 steps short-response"
```

This notebook mirrors that process inside Kaggle's environment using the adapter
published as a dataset input.


## 1. Locate adapter from dataset input

In [ ]:
import os, shutil, json

# Adapter published as Kaggle dataset: gdataranger/nemotron-v05-sft-unsloth
# Add via notebook UI: Input → + Add Input → Datasets → nemotron-v05-sft-unsloth
DATASET_INPUT = "/kaggle/input/nemotron-v05-sft-unsloth"
OUTPUT_DIR    = "/kaggle/working"

if os.path.isdir(DATASET_INPUT):
    print(f"Found adapter dataset at: {DATASET_INPUT}")
    print("Files:", os.listdir(DATASET_INPUT))
    ADAPTER_SOURCE = DATASET_INPUT
else:
    print(f"Dataset input not found at {DATASET_INPUT}")
    print("Add the dataset via: Input → + Add Input → gdataranger/nemotron-v05-sft-unsloth")
    print("Falling back to listing /kaggle/input for available inputs:")
    if os.path.isdir("/kaggle/input"):
        for d in os.listdir("/kaggle/input"):
            print(f"  /kaggle/input/{d}")
    ADAPTER_SOURCE = None


## 2. Copy adapter files to /kaggle/working

In [ ]:
REQUIRED = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "chat_template.jinja",
]
OPTIONAL = ["special_tokens_map.json"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

if ADAPTER_SOURCE is None:
    print("ERROR: No adapter source found. Add gdataranger/nemotron-v05-sft-unsloth as input.")
else:
    for fname in REQUIRED + OPTIONAL:
        src = os.path.join(ADAPTER_SOURCE, fname)
        dst = os.path.join(OUTPUT_DIR, fname)
        if os.path.exists(src):
            shutil.copy2(src, dst)
            size = os.path.getsize(dst)
            print(f"  {fname}  ({size/1024/1024:.1f} MB)")
        elif fname in REQUIRED:
            print(f"  MISSING (required): {fname}")
        else:
            print(f"  skipped (optional): {fname}")
    print(f"\nAdapter ready in {OUTPUT_DIR}")


## 3. Verify competition constraints

In [ ]:
config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")

if not os.path.exists(config_path):
    print("adapter_config.json not found — run cell 2 first")
else:
    with open(config_path) as f:
        config = json.load(f)

    r     = config.get("r", "?")
    base  = config.get("base_model_name_or_path", "?")
    mods  = config.get("target_modules", [])
    print(f"base_model:     {base}")
    print(f"lora_r:         {r}  (limit: 32)")
    print(f"target_modules: {mods}")
    assert isinstance(r, int) and r <= 32, f"LoRA rank {r} exceeds competition limit of 32"
    assert "NVIDIA-Nemotron-3-Nano-30B" in base, f"Wrong base model: {base}"
    print("\nCompetition constraints: OK ✓")

    print("\nFiles in /kaggle/working:")
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            print(f"  {fname}  ({os.path.getsize(fpath)/1024/1024:.1f} MB)")
